In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

no null values obtained

In [5]:
import os
import pandas as pd

# 1. Path to the folder containing CSV files
FOLDER_PATH = 'D:/coding/financial-data-analysis/ohlcv_data'

# 2. Optional: store processed results
null_values = []

# 3. Loop through all files in the folder
for filename in os.listdir(FOLDER_PATH):
    if filename.endswith(".csv"):
        file_path = os.path.join(FOLDER_PATH, filename)
        
        # 4. Read CSV
        df = pd.read_csv(file_path)
        if df.isnull().any().any():
            null_values.append(filename)


        
        
null_values


[]

In [2]:
def calculate_mean_std_z_skew_kurt(df, window):
    roll = df['Returns'].rolling(window=window)

    df[f'Mean_{window}'] = roll.mean()
    df[f'Std_{window}'] = roll.std()
    df[f'Z_Score_{window}'] = (df['Returns'] - df[f'Mean_{window}']) / df[f'Std_{window}']
    df[f'Skew_{window}'] = roll.skew()
    df[f'Kurt_{window}'] = roll.kurt()   

    return (
        df[f'Mean_{window}'],
        df[f'Std_{window}'],
        df[f'Z_Score_{window}'],
        df[f'Skew_{window}'],
        df[f'Kurt_{window}']
    )


def rolling_parkinson_vol(df, window):
    hl_log_sq = np.log(df['High'] / df['Low']) ** 2
    vol = np.sqrt(
        hl_log_sq.rolling(window).mean() / (4 * np.log(2))
    )
    return vol


def rolling_garman_klass_vol(df, window):
    hl = np.log(df['High'] / df['Low']) ** 2
    co = np.log(df['Close'] / df['Open']) ** 2

    vol = np.sqrt(
        (0.5 * hl - (2 * np.log(2) - 1) * co).rolling(window).mean()
    )
    return vol


def rolling_yang_zhang_vol(df, window):
    log_ho = np.log(df['High'] / df['Open'])
    log_lo = np.log(df['Low'] / df['Open'])
    log_co = np.log(df['Close'] / df['Open'])
    log_oc = np.log(df['Open'] / df['Close'].shift(1))

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

    sigma_oc = log_oc.rolling(window).var()
    sigma_co = log_co.rolling(window).var()
    sigma_rs = rs.rolling(window).mean()

    k = 0.34 / (1.34 + (window + 1) / (window - 1))

    vol = np.sqrt(sigma_oc + k * sigma_co + (1 - k) * sigma_rs)
    return vol


In [ ]:
FOLDER_PATH = 'D:/coding/financial-data-analysis/ohlcv_data'

important_stats = []

for filename in os.listdir(FOLDER_PATH):
    if filename.endswith(".csv"):
        file_path = os.path.join(FOLDER_PATH, filename)
        
        #standardized features
        df = pd.read_csv(file_path)
        df['OH_ratio'] = (df['Open'] - df['High']) / df['Open']
        df['OL_ratio'] = (df['Open'] - df['Low']) / df['Open']
        df['CH_ratio'] = (df['Close'] - df['High']) / df['Close']
        df['CL_ratio'] = (df['Close'] - df['Low']) / df['Close']
        df['Price_Change'] = (df['Close'] - df['Open']) / df['Open']
        df['Returns'] = df['Close'].pct_change()
        df['Day_Range'] = (df['High'] - df['Low']) / df['Low']

        #basic statistical features
        for w in [5, 20, 50, 100, 200]:
            (df[f'Mean_{w}'],
             df[f'Std_{w}'],
             df[f'Z_score_{w}'],
             df[f'Skew_{w}'],
             df[f'Kurt_{w}']) = calculate_mean_std_z_skew_kurt(df, window=w)


      
      # volatility based features
        for w in [5, 20, 50, 100, 200]:
            df[f'Parkinson_Vol_{w}'] = rolling_parkinson_vol(df, w)
            df[f'GK_Vol_{w}'] = rolling_garman_klass_vol(df, w)
            df[f'YZ_Vol_{w}'] = rolling_yang_zhang_vol(df, w)
  
